In [1]:
# ==========================================
# 1. INSTALLATIONS & IMPORTS
# ==========================================
# Install necessary libraries (Quiet mode)
!pip install -q transformers datasets torch scikit-learn accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    BertPreTrainedModel,
    BertModel,
    Trainer,
    TrainingArguments,
    EvalPrediction
)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. CONFIGURATION
# ==========================================
MODEL_ID = "bert-large-uncased"
MAX_LENGTH = 256        # Length of (Question + 1 Chunk)
CHUNK_STRIDE = 160       # Overlap
MAX_CHUNKS = 16       # Max chunks per answer
BATCH_SIZE = 8
FOCAL_GAMMA = 2.0       # Focusing parameter

# ==========================================
# 3. DATA LOADING & WEIGHT CALCULATION
# ==========================================
print("\n--- Loading Data ---")
dataset = load_dataset("ailsntua/QEvasion")

# Map Labels
labels_list = dataset['train'].unique('clarity_label')
num_labels = len(labels_list)
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}

print(f"Labels: {label2id}")

def encode_labels(batch):
    return {"labels": label2id[batch['clarity_label']]}

dataset = dataset.map(encode_labels)

# Calculate Class Weights
print("\n--- Calculating Class Weights ---")
train_labels = dataset['train']['labels']
label_counts = Counter(train_labels)
total_samples = len(train_labels)

class_weights = []
for i in range(num_labels):
    count = label_counts.get(i, 1)
    weight = total_samples / (num_labels * count)
    class_weights.append(weight)

# Convert to Tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class Weights: {class_weights_tensor}")

# ==========================================
# 4. CUSTOM DATASET (THE CHUNKER)
# ==========================================
class MILChunkingDataset(Dataset):
    """
    Takes a row (Question, Long Answer) and splits it into multiple
    overlapping chunks ON THE FLY.
    """
    def __init__(self, hf_dataset, tokenizer, max_len, stride, max_chunks):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        question = row['question']
        long_answer = row['interview_answer']
        label = row['labels']

        # 1. Tokenize answer without truncation
        answer_tokens = self.tokenizer(long_answer, add_special_tokens=False)['input_ids']

        # Calculate space available for answer tokens (Total - Question - Special Tokens)
        # BERT uses [CLS] Q [SEP] A [SEP], so approx 3 special tokens + Q length
        # We leave a buffer of 64 tokens for the Question
        tokens_per_chunk = self.max_len - 64

        if len(answer_tokens) == 0:
            windows = [[]]
        else:
            windows = [
                answer_tokens[i : i + tokens_per_chunk]
                for i in range(0, len(answer_tokens), self.stride)
            ]
            windows = windows[:self.max_chunks]

        # 2. Pair EACH window with the Question
        chunk_input_ids = []
        chunk_attention_masks = []
        chunk_token_type_ids = [] # BERT needs token_type_ids (segment IDs)

        for window in windows:
            window_text = self.tokenizer.decode(window)

            # [CLS] Question [SEP] Window [SEP]
            encoded = self.tokenizer(
                question,
                window_text,
                padding='max_length',
                truncation=True,
                max_length=self.max_len,
                return_tensors='pt'
            )
            chunk_input_ids.append(encoded['input_ids'].squeeze(0))
            chunk_attention_masks.append(encoded['attention_mask'].squeeze(0))
            chunk_token_type_ids.append(encoded['token_type_ids'].squeeze(0))

        # 3. Stack
        return {
            "input_ids": torch.stack(chunk_input_ids),       # [N_Chunks, Seq_Len]
            "attention_mask": torch.stack(chunk_attention_masks),
            "token_type_ids": torch.stack(chunk_token_type_ids),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 5. DATA COLLATOR (THE PADDER)
# ==========================================
@dataclass
class MILDataCollator:
    """
    Pads the 'Bag' dimension so all items in a batch have the same number of chunks.
    """
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_chunks_batch = max(f["input_ids"].shape[0] for f in features)

        batch_input_ids = []
        batch_masks = []
        batch_token_types = []
        batch_labels = []

        for f in features:
            source_ids = f["input_ids"]
            source_mask = f["attention_mask"]
            source_types = f["token_type_ids"]
            num_chunks, seq_len = source_ids.shape

            pad_chunks = max_chunks_batch - num_chunks
            if pad_chunks > 0:
                pad_ids = torch.zeros((pad_chunks, seq_len), dtype=torch.long)
                pad_mask = torch.zeros((pad_chunks, seq_len), dtype=torch.long)
                pad_types = torch.zeros((pad_chunks, seq_len), dtype=torch.long)

                padded_ids = torch.cat([source_ids, pad_ids], dim=0)
                padded_mask = torch.cat([source_mask, pad_mask], dim=0)
                padded_types = torch.cat([source_types, pad_types], dim=0)
            else:
                padded_ids = source_ids
                padded_mask = source_mask
                padded_types = source_types

            batch_input_ids.append(padded_ids)
            batch_masks.append(padded_mask)
            batch_token_types.append(padded_types)
            batch_labels.append(f["labels"])

        return {
            "input_ids": torch.stack(batch_input_ids), # [Batch, Max_Chunks, Seq_Len]
            "attention_mask": torch.stack(batch_masks),
            "token_type_ids": torch.stack(batch_token_types),
            "labels": torch.stack(batch_labels)
        }

# ==========================================
# 6. CUSTOM MODEL (BERT + ATTENTION MIL + FOCAL LOSS)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        pt = torch.clamp(pt, min=1e-8, max=1.0 - 1e-8)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class BertForMIL(BertPreTrainedModel):
    def __init__(self, config, class_weights=None, focal_gamma=2.0):
        super().__init__(config)
        self.num_labels = config.num_labels

        # Standard BERT Body
        self.bert = BertModel(config)

        # Attention Mechanism
        # Projects hidden size to a scalar attention score
        self.attention_layer = nn.Linear(config.hidden_size, 1)

        # Classifier Head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Loss Setup
        self.class_weights = class_weights
        self.focal_gamma = focal_gamma
        self.post_init()

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        **kwargs
    ) -> SequenceClassifierOutput:

        # input_ids: [Batch, Chunks, Seq_Len]
        batch_size, num_chunks, seq_len = input_ids.shape

        # 1. Flatten: [Batch * Chunks, Seq_Len]
        flat_input_ids = input_ids.view(-1, seq_len)
        flat_mask = attention_mask.view(-1, seq_len)
        flat_token_types = token_type_ids.view(-1, seq_len) if token_type_ids is not None else None

        # 2. Pass through BERT
        outputs = self.bert(
            input_ids=flat_input_ids,
            attention_mask=flat_mask,
            token_type_ids=flat_token_types
        )

        # 3. Extract [CLS] Token (First token)
        # Shape: [Batch * Chunks, Hidden_Size]
        cls_output = outputs.last_hidden_state[:, 0, :]

        # --- ATTENTION POOLING START ---

        # Calculate raw attention scores
        # Shape: [Batch * Chunks, 1]
        attn_scores = self.attention_layer(cls_output)

        # Reshape to [Batch, Chunks]
        attn_scores = attn_scores.view(batch_size, num_chunks)

        # Mask padding chunks (where attention_mask is 0 for the whole chunk)
        # We check if a chunk is "real" by seeing if it has ANY non-pad tokens in the seq dim
        # Shape: [Batch, Chunks]
        chunk_mask = torch.any(attention_mask > 0, dim=-1)

        # Set attention scores of padding chunks to -infinity (so softmax is 0)
        min_val = -65000.0
        attn_scores = attn_scores.masked_fill(~chunk_mask, min_val)

        # Apply Softmax to get probability distribution over chunks
        # Shape: [Batch, Chunks]
        attn_weights = F.softmax(attn_scores, dim=1)

        # Reshape cls_output back to [Batch, Chunks, Hidden]
        cls_output_reshaped = cls_output.view(batch_size, num_chunks, -1)

        # Compute Weighted Average
        # [Batch, Chunks, 1] * [Batch, Chunks, Hidden] -> sum dim 1 -> [Batch, Hidden]
        context_vector = torch.sum(cls_output_reshaped * attn_weights.unsqueeze(-1), dim=1)

        # --- ATTENTION POOLING END ---

        # 4. Classifier Head
        context_vector = self.dropout(context_vector)
        logits = self.classifier(context_vector)

        loss = None
        if labels is not None:
            if self.class_weights is not None:
                self.class_weights = self.class_weights.to(logits.device)

            # Compute Loss
            loss_fct = FocalLoss(gamma=self.focal_gamma, weight=self.class_weights)
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# ==========================================
# 7. TRAINING EXECUTION
# ==========================================

# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Create Datasets
print("\n--- Creating Datasets ---")
train_ds = MILChunkingDataset(dataset['train'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS)
test_ds = MILChunkingDataset(dataset['test'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS)

# Initialize Model
print("\n--- Initializing BERT Model ---")
model = BertForMIL.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    class_weights=class_weights_tensor,
    focal_gamma=FOCAL_GAMMA
)

# Metrics
def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    f1 = f1_score(p.label_ids, preds, average='macro')
    return {"accuracy": acc, "f1_macro": f1}

# Training Arguments
training_args = TrainingArguments(
    output_dir="./Bert_MIL_Attention",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=MILDataCollator(),
    compute_metrics=compute_metrics
)

# Start Training
print("\n--- Starting Training ---")
trainer.train()

# ==========================================
# 8. FINAL EVALUATION
# ==========================================
print("\n--- Final Evaluation on Test Set ---")
results = trainer.predict(test_ds)
y_preds = np.argmax(results.predictions, axis=1)
y_true = results.label_ids

print("\nClassification Report:")
print(classification_report(y_true, y_preds, target_names=labels_list))

# Save
trainer.save_model("./Final_Bert_MIL_Attn_Model")
tokenizer.save_pretrained("./Final_Bert_MIL_Attn_Model")
print("\nModel saved to ./Final_Bert_MIL_Attn_Model")

Using device: cuda

--- Loading Data ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Some weights of BertForMIL were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['attention_layer.bias', 'attention_layer.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Token indices sequence length is longer than the specified maximum sequence length for this model (866 > 512). Running this sequence through the model will result in indexing errors



--- Starting Training ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.539400,0.295278,0.298701,0.300492
2,0.268300,0.358400,0.558442,0.552905
3,0.204700,0.411305,0.584416,0.577886
4,0.090800,0.659430,0.665584,0.617224
5,0.033300,0.842377,0.688312,0.586947
6,0.012100,0.837662,0.694805,0.620776
7,0.026400,0.932866,0.707792,0.639162
8,0.002300,0.988122,0.707792,0.636591
9,0.004200,1.017923,0.714286,0.646919
10,0.004400,1.019311,0.717532,0.653104



--- Final Evaluation on Test Set ---



Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.55      0.58      0.56        79
     Ambivalent       0.79      0.79      0.79       206
Clear Non-Reply       0.65      0.57      0.60        23

       accuracy                           0.72       308
      macro avg       0.66      0.64      0.65       308
   weighted avg       0.72      0.72      0.72       308


Model saved to ./Final_Bert_MIL_Attn_Model
